# Dynamic Programming — Study Guide

**Learning order:**

```
Part 0 — What is DP & The Universal Framework
  └── When to use DP → Top-down vs Bottom-up → The 5-step framework
Part 1 — 1D DP Patterns
  └── Linear DP → Kadane's → Fibonacci-style
Part 2 — 2D DP Patterns
  └── Grid DP → String DP (LCS, Edit Distance)
Part 3 — Knapsack Patterns
  └── 0/1 Knapsack → Unbounded Knapsack → Subset Sum
Part 4 — Interval & Advanced DP
  └── Interval DP → DP on Trees
Part 5 — Decision Guide
```

Each section introduces the concept first, then shows the pseudo-code immediately after.

---
# Part 0 — What is DP & The Universal Framework

## What is Dynamic Programming?

DP is an optimization technique for problems that have two properties:

1. **Overlapping subproblems** — the same subproblem is solved multiple times in a naive recursion
2. **Optimal substructure** — the optimal solution to the whole problem can be built from optimal solutions to subproblems

The core idea is simple: **solve each subproblem once and store the result** so you never compute it again.

---
## When to use DP

Look for these signal words and patterns in the problem:

| Signal | Example |
| :--- | :--- |
| "maximum / minimum" of something | Maximum profit, minimum cost, minimum steps |
| "number of ways" | How many ways to climb stairs, how many distinct paths |
| "can you reach / is it possible" | Can you reach the last index, can the sum be achieved |
| "longest / shortest" subsequence or substring | Longest common subsequence, longest increasing subsequence |
| Decision at each step affects future decisions | Buy/sell stock, take/skip an item |

**Quick test:** Can you draw a recursion tree where the same node appears more than once? If yes → DP.

---
## The 5-Step Framework

Apply these five steps in order for every DP problem. Writing them as comments before coding saves enormous time.

```
Step 1 — Define the state
         What does dp[i] (or dp[i][j]) represent?
         Be precise: "dp[i] = maximum profit using the first i items"

Step 2 — Find the transition (recurrence)
         How do I compute dp[i] from smaller subproblems?
         e.g. dp[i] = max(dp[i-1], dp[i-2] + nums[i])

Step 3 — Identify the base case
         What are the smallest subproblems I can answer directly?
         e.g. dp[0] = 0, dp[1] = nums[0]

Step 4 — Determine the order of computation
         Which direction do I fill the table?
         Usually left-to-right, top-to-bottom

Step 5 — Extract the final answer
         Where in dp is the answer?
         e.g. dp[n], max(dp), dp[n][m]
```

---
## Top-down vs Bottom-up — Two Implementation Styles

Every DP problem can be implemented in either style. They always produce the same answer.

| | Top-down (Recursion + Memo) | Bottom-up (Iterative Table) |
| :--- | :--- | :--- |
| **How it works** | Recurse naturally, cache results | Fill a table from base case upward |
| **Code style** | Recursive function + `memo` dict | Loops + `dp` array |
| **Ease of writing** | Easier — mirrors the recurrence directly | Requires knowing the fill order |
| **Stack overflow risk** | Yes — deep recursion on large inputs | No |
| **Space optimization** | Harder | Easy — can reduce to O(1) by reusing rows |
| **When to prefer** | When the state space is sparse (not all states visited) | When you need maximum performance |

In [ ]:
# Example: LC 70 — Climbing Stairs (can take 1 or 2 steps)
# dp[i] = number of ways to reach step i

# ── Top-down (recursion + memo) ──────────────────────────────────────────────
def climbStairs_topdown(n):
    memo = {}

    def dp(i):
        if i <= 1: return 1                 # base case
        if i in memo: return memo[i]        # return cached result
        memo[i] = dp(i - 1) + dp(i - 2)    # recurrence
        return memo[i]

    return dp(n)


# ── Bottom-up (iterative table) ───────────────────────────────────────────────
def climbStairs_bottomup(n):
    if n <= 1: return 1
    dp = [0] * (n + 1)
    dp[0], dp[1] = 1, 1             # base cases
    for i in range(2, n + 1):
        dp[i] = dp[i-1] + dp[i-2]  # recurrence
    return dp[n]


# ── Space-optimized bottom-up (O(1) space) ────────────────────────────────────
def climbStairs_optimized(n):
    if n <= 1: return 1
    prev2, prev1 = 1, 1
    for _ in range(2, n + 1):
        prev2, prev1 = prev1, prev1 + prev2
    return prev1

---
# Part 1 — 1D DP Patterns

The simplest DP problems: `dp` is a 1D array and each state depends on a fixed number of previous states.

---
## Type 1 — Linear / Fibonacci-style DP

**Pattern:** `dp[i]` depends on `dp[i-1]`, `dp[i-2]`, or a small fixed window of previous values.

**Signal words:** "number of ways", "reach the end", "minimum cost to reach".

**Key insight:** if `dp[i]` only depends on the last 1–2 values, you can reduce space from O(n) to O(1) using two variables.

In [ ]:
# Template — linear 1D DP
def linear_dp(nums):
    n = len(nums)
    dp = [0] * n
    dp[0] = BASE_CASE
    for i in range(1, n):
        dp[i] = TRANSITION(dp[i-1], dp[i-2], ...)  # depends on previous states
    return dp[n-1]  # or max(dp), sum(dp), etc.


# Example: LC 746 — Min Cost Climbing Stairs
# dp[i] = minimum cost to reach step i
def minCostClimbingStairs(cost):
    n = len(cost)
    dp = [0] * (n + 1)
    for i in range(2, n + 1):
        dp[i] = min(dp[i-1] + cost[i-1],
                    dp[i-2] + cost[i-2])
    return dp[n]


# Example: LC 198 — House Robber
# dp[i] = maximum money robbing from houses 0..i
# Cannot rob two adjacent houses
def rob(nums):
    if not nums: return 0
    if len(nums) == 1: return nums[0]
    dp = [0] * len(nums)
    dp[0] = nums[0]
    dp[1] = max(nums[0], nums[1])
    for i in range(2, len(nums)):
        dp[i] = max(dp[i-1],            # skip house i
                    dp[i-2] + nums[i])  # rob house i
    return dp[-1]

**Practice problems:**

| Problem | State definition | Transition |
| :--- | :--- | :--- |
| LC 70 — Climbing Stairs | `dp[i]` = ways to reach step i | `dp[i-1] + dp[i-2]` |
| LC 198 — House Robber | `dp[i]` = max money from first i houses | `max(dp[i-1], dp[i-2] + nums[i])` |
| LC 746 — Min Cost Climbing Stairs | `dp[i]` = min cost to reach step i | `min(dp[i-1] + cost[i-1], dp[i-2] + cost[i-2])` |

---
## Type 2 — Kadane's Algorithm (Maximum Subarray)

**Pattern:** at each position, decide whether to **extend** the current subarray or **start fresh**.

**Signal words:** "maximum sum subarray", "contiguous subarray".

**Key insight:** `dp[i]` = max subarray ending exactly at index i. If `dp[i-1] < 0`, starting fresh is always better.

In [ ]:
# LC 53 — Maximum Subarray (Kadane's Algorithm)
# dp[i] = maximum sum of subarray ending at index i
def maxSubArray(nums):
    dp = [0] * len(nums)
    dp[0] = nums[0]
    for i in range(1, len(nums)):
        dp[i] = max(nums[i],            # start fresh at i
                    dp[i-1] + nums[i])  # extend previous subarray
    return max(dp)


# Space-optimized version — O(1) space
def maxSubArray_optimized(nums):
    cur_sum = res = nums[0]
    for num in nums[1:]:
        cur_sum = max(num, cur_sum + num)
        res = max(res, cur_sum)
    return res

**Practice problems:**

| Problem | Notes |
| :--- | :--- |
| LC 53 — Maximum Subarray | Classic Kadane's |
| LC 152 — Maximum Product Subarray | Track both max and min (negative × negative = positive) |
| LC 918 — Maximum Sum Circular Subarray | Answer = max(normal Kadane, total sum − min subarray) |

---
## Type 3 — Longest Increasing Subsequence (LIS)

**Pattern:** for each element, look back at all previous elements and extend the best valid subsequence.

**Signal words:** "longest increasing subsequence", "longest non-decreasing", "minimum number of chains".

**Key insight:** `dp[i]` = length of LIS ending at index i. Answer is `max(dp)`, not `dp[-1]`.

In [ ]:
# LC 300 — Longest Increasing Subsequence
# dp[i] = length of LIS ending at index i
def lengthOfLIS(nums):
    n = len(nums)
    dp = [1] * n                        # every element is a subsequence of length 1
    for i in range(1, n):
        for j in range(i):              # look back at all previous elements
            if nums[j] < nums[i]:       # valid extension
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)                      # answer is the max over all ending positions


# O(n log n) version using binary search + patience sorting
import bisect
def lengthOfLIS_fast(nums):
    tails = []                          # tails[i] = smallest tail of all LIS of length i+1
    for num in nums:
        pos = bisect.bisect_left(tails, num)
        if pos == len(tails):
            tails.append(num)           # extend LIS
        else:
            tails[pos] = num            # replace with smaller tail
    return len(tails)

**Practice problems:**

| Problem | Notes |
| :--- | :--- |
| LC 300 — Longest Increasing Subsequence | Core LIS — start here |
| LC 354 — Russian Doll Envelopes | Sort by width ASC, height DESC → LIS on height |
| LC 673 — Number of LIS | Track both length and count arrays |

---
### 1D DP — Question Type Summary

| Type | Signal words | Key idea | Problems |
| :--- | :--- | :--- | :--- |
| Linear / Fibonacci | "ways to reach", "min cost" | `dp[i]` from fixed previous window | 70, 198, 746 |
| Kadane's | "maximum sum subarray" | Extend or start fresh at each step | 53, 152, 918 |
| LIS | "longest increasing subsequence" | Look back at all previous, extend best | 300, 354, 673 |

---
# Part 2 — 2D DP Patterns

`dp` becomes a 2D table. The state needs two dimensions — usually position in two sequences, or row/column in a grid.

---
## Type 4 — Grid DP

**Pattern:** navigate a 2D grid, making decisions at each cell. `dp[i][j]` = best result reaching cell `(i, j)`.

**Signal words:** "unique paths", "minimum path sum", "reach bottom-right corner".

**Key insight:** initialize the first row and column as base cases (can only come from one direction), then fill the rest using transitions from top and left.

In [ ]:
# Template — Grid DP
def grid_dp(grid):
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]

    dp[0][0] = grid[0][0]               # base case: top-left cell

    for i in range(1, m):               # first column — can only come from above
        dp[i][0] = dp[i-1][0] + grid[i][0]
    for j in range(1, n):               # first row — can only come from left
        dp[0][j] = dp[0][j-1] + grid[0][j]

    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = min(dp[i-1][j], dp[i][j-1]) + grid[i][j]  # from top or left

    return dp[m-1][n-1]


# Example: LC 62 — Unique Paths
# dp[i][j] = number of unique paths to reach cell (i, j)
def uniquePaths(m, n):
    dp = [[1] * n for _ in range(m)]    # base case: first row and col = 1
    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = dp[i-1][j] + dp[i][j-1]
    return dp[m-1][n-1]


# Example: LC 64 — Minimum Path Sum
# dp[i][j] = minimum sum path to reach cell (i, j)
def minPathSum(grid):
    m, n = len(grid), len(grid[0])
    dp = [[0]*n for _ in range(m)]
    dp[0][0] = grid[0][0]
    for i in range(1, m): dp[i][0] = dp[i-1][0] + grid[i][0]
    for j in range(1, n): dp[0][j] = dp[0][j-1] + grid[0][j]
    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = min(dp[i-1][j], dp[i][j-1]) + grid[i][j]
    return dp[m-1][n-1]

**Practice problems:**

| Problem | State definition | Notes |
| :--- | :--- | :--- |
| LC 62 — Unique Paths | `dp[i][j]` = paths to `(i,j)` | Base case: all 1s in row 0 and col 0 |
| LC 63 — Unique Paths II | `dp[i][j]` = paths to `(i,j)` | Set `dp[i][j] = 0` if obstacle |
| LC 64 — Minimum Path Sum | `dp[i][j]` = min sum to `(i,j)` | Take min from top or left |

---
## Type 5 — String DP (LCS & Edit Distance)

**Pattern:** two strings as input. `dp[i][j]` = answer for the first `i` chars of string 1 and first `j` chars of string 2.

**Signal words:** "longest common subsequence", "edit distance", "shortest common supersequence", "delete operations".

**Key insight:** the transition always branches on whether `s1[i-1] == s2[j-1]`:
- If characters match → extend the diagonal
- If they don't → take the best of top, left, or diagonal (depending on problem)

In [ ]:
# Example: LC 1143 — Longest Common Subsequence (LCS)
# dp[i][j] = LCS length of s1[:i] and s2[:j]
def longestCommonSubsequence(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]  # extra row/col as base case (empty string)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:              # characters match → extend diagonal
                dp[i][j] = dp[i-1][j-1] + 1
            else:                                # skip one character from either string
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]


# Example: LC 72 — Edit Distance
# dp[i][j] = min operations to convert s1[:i] to s2[:j]
# Operations: insert, delete, replace
def minDistance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1): dp[i][0] = i  # delete all chars from s1
    for j in range(n + 1): dp[0][j] = j  # insert all chars of s2

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:          # characters match — no operation needed
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],              # delete from s1
                    dp[i][j-1],              # insert into s1
                    dp[i-1][j-1]             # replace
                )
    return dp[m][n]

**Practice problems:**

| Problem | State definition | Match transition | No-match transition |
| :--- | :--- | :--- | :--- |
| LC 1143 — LCS | `dp[i][j]` = LCS of `s1[:i]`, `s2[:j]` | `dp[i-1][j-1] + 1` | `max(dp[i-1][j], dp[i][j-1])` |
| LC 72 — Edit Distance | `dp[i][j]` = min edits | `dp[i-1][j-1]` | `1 + min(top, left, diagonal)` |
| LC 583 — Delete Operations | `dp[i][j]` = min deletions | `dp[i-1][j-1]` | `1 + min(dp[i-1][j], dp[i][j-1])` |

---
### 2D DP — Question Type Summary

| Type | Signal words | Key idea | Problems |
| :--- | :--- | :--- | :--- |
| Grid DP | "unique paths", "min path sum" | Fill from top-left; init first row/col | 62, 63, 64 |
| String DP | "LCS", "edit distance", "common subsequence" | Branch on character match; extra row/col for empty string | 1143, 72, 583 |

---
# Part 3 — Knapsack Patterns

Knapsack problems are the most important DP family. Master these and you can solve a large class of optimization problems.

**Core setup:** you have a set of items, each with a weight and value. You have a capacity. Maximize value without exceeding capacity.

---
## Type 6 — 0/1 Knapsack

**Rule:** each item can be used **at most once** (take it or skip it).

**Signal words:** "subset sum", "partition equal subset", "target sum", "can you pick items to reach exactly X".

**Key insight:** iterate items in the outer loop, capacity in the inner loop. **Iterate capacity backwards** to prevent reusing the same item.

In [ ]:
# Template — 0/1 Knapsack
# dp[j] = max value achievable with capacity j
def knapsack_01(weights, values, capacity):
    dp = [0] * (capacity + 1)
    for i in range(len(weights)):           # for each item
        for j in range(capacity, weights[i]-1, -1):  # iterate BACKWARDS
            dp[j] = max(dp[j],              # skip item i
                        dp[j - weights[i]] + values[i])  # take item i
    return dp[capacity]


# Example: LC 416 — Partition Equal Subset Sum
# Can we split nums into two subsets with equal sum?
# → Is there a subset that sums to total/2?
# dp[j] = True if subset summing to j exists
def canPartition(nums):
    total = sum(nums)
    if total % 2 != 0: return False
    target = total // 2

    dp = [False] * (target + 1)
    dp[0] = True                            # empty subset sums to 0

    for num in nums:
        for j in range(target, num - 1, -1):  # iterate BACKWARDS
            dp[j] = dp[j] or dp[j - num]

    return dp[target]


# Example: LC 494 — Target Sum
# Assign + or - to each number to reach target
# dp[j] = number of ways to reach sum j
def findTargetSumWays(nums, target):
    total = sum(nums)
    # math reduction: let P = positive subset sum
    # P - (total - P) = target → P = (total + target) / 2
    if (total + target) % 2 != 0: return 0
    s = (total + target) // 2

    dp = [0] * (s + 1)
    dp[0] = 1
    for num in nums:
        for j in range(s, num - 1, -1):     # iterate BACKWARDS
            dp[j] += dp[j - num]
    return dp[s]

**Practice problems:**

| Problem | Knapsack framing | Notes |
| :--- | :--- | :--- |
| LC 416 — Partition Equal Subset Sum | Can subset reach `total/2`? | Boolean DP |
| LC 494 — Target Sum | Count subsets reaching `(total+target)/2` | Count DP |
| LC 474 — Ones and Zeroes | 2D knapsack (m zeros, n ones as capacity) | Two-dimensional capacity |

---
## Type 7 — Unbounded Knapsack

**Rule:** each item can be used **unlimited times**.

**Signal words:** "coin change", "infinite supply", "complete", "any number of times".

**Key insight:** iterate capacity **forwards** (not backwards) — this allows the same item to be reused.

In [ ]:
# Template — Unbounded Knapsack
# Iterate capacity FORWARDS to allow reuse
def knapsack_unbounded(items, capacity):
    dp = [0] * (capacity + 1)
    for i in range(len(items)):
        for j in range(items[i], capacity + 1):  # iterate FORWARDS
            dp[j] = max(dp[j], dp[j - items[i]] + values[i])
    return dp[capacity]


# Example: LC 322 — Coin Change
# dp[j] = minimum coins needed to make amount j
def coinChange(coins, amount):
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0                               # base case: 0 coins for amount 0
    for coin in coins:
        for j in range(coin, amount + 1):   # iterate FORWARDS (unbounded)
            dp[j] = min(dp[j], dp[j - coin] + 1)
    return dp[amount] if dp[amount] != float('inf') else -1


# Example: LC 518 — Coin Change II
# dp[j] = number of combinations to make amount j
def change(amount, coins):
    dp = [0] * (amount + 1)
    dp[0] = 1                               # one way to make 0: use no coins
    for coin in coins:
        for j in range(coin, amount + 1):   # iterate FORWARDS (unbounded)
            dp[j] += dp[j - coin]
    return dp[amount]

**Practice problems:**

| Problem | dp meaning | Notes |
| :--- | :--- | :--- |
| LC 322 — Coin Change | min coins for amount j | Init with `inf`, answer is `dp[amount]` |
| LC 518 — Coin Change II | number of combinations for j | Init `dp[0]=1` |
| LC 139 — Word Break | can string be segmented? | `dp[j] = True` if `s[:j]` can be built from wordDict |

---
### 0/1 vs Unbounded — The One Rule That Matters

```
Each item usable ONCE  → iterate capacity BACKWARDS   (0/1 knapsack)
Each item usable MANY  → iterate capacity FORWARDS     (unbounded knapsack)
```

Everything else in the template stays the same.

---
### Knapsack — Question Type Summary

| Type | Each item | Capacity direction | Signal words | Problems |
| :--- | :--- | :--- | :--- | :--- |
| 0/1 Knapsack | Once | Backwards | "subset", "partition", "target sum" | 416, 494, 474 |
| Unbounded Knapsack | Unlimited | Forwards | "coin change", "infinite supply" | 322, 518, 139 |

---
# Part 4 — Interval & Advanced DP

---
## Type 8 — Interval DP

**Pattern:** the problem is defined on a contiguous range `[i, j]` of elements. You split the range at some midpoint `k` and combine results from `[i, k]` and `[k+1, j]`.

**Signal words:** "burst balloons", "matrix chain multiplication", "minimum cost to merge", "score from splitting".

**Key insight:**
- `dp[i][j]` = answer for the subarray/range from index i to j
- Fill by **increasing length** (small ranges first, then build up to the full range)
- The final answer is `dp[0][n-1]`

In [ ]:
# Template — Interval DP
def interval_dp(arr):
    n = len(arr)
    dp = [[0] * n for _ in range(n)]

    # Fill by increasing length — small intervals first
    for length in range(2, n + 1):          # interval length
        for i in range(n - length + 1):     # start of interval
            j = i + length - 1             # end of interval
            for k in range(i, j):           # split point
                dp[i][j] = max(dp[i][j],
                               dp[i][k] + dp[k+1][j] + COST(i, k, j))

    return dp[0][n-1]


# Example: LC 312 — Burst Balloons
# dp[i][j] = max coins from bursting all balloons in (i, j) exclusive
# Trick: think of k as the LAST balloon to burst in range (i, j)
def maxCoins(nums):
    nums = [1] + nums + [1]                 # add boundary balloons
    n = len(nums)
    dp = [[0] * n for _ in range(n)]

    for length in range(2, n):
        for i in range(0, n - length):
            j = i + length
            for k in range(i + 1, j):       # k = last balloon to burst
                dp[i][j] = max(dp[i][j],
                               dp[i][k] + nums[i]*nums[k]*nums[j] + dp[k][j])
    return dp[0][n-1]

**Practice problems:**

| Problem | State definition | Notes |
| :--- | :--- | :--- |
| LC 312 — Burst Balloons | `dp[i][j]` = max coins in open interval `(i,j)` | Think of k as the **last** to burst, not first |
| LC 1312 — Min Insertions to Make Palindrome | `dp[i][j]` = min insertions for `s[i..j]` | Similar to edit distance on a single string |
| LC 1039 — Minimum Score Triangulation | `dp[i][j]` = min score triangulating polygon `[i..j]` | Split at vertex k |

---
## Type 9 — DP on Trees

**Pattern:** DP where the subproblems follow the tree structure. Each node's result depends on its children's results — naturally a postorder DFS.

**Signal words:** "max sum with no two adjacent nodes", "minimum vertex cover", "tree diameter".

**Key insight:** the state usually includes two choices at each node: **include this node** or **exclude this node**. Return both from each DFS call.

In [ ]:
# Template — DP on Trees
# Returns (result_if_included, result_if_excluded) for each node
def dp_tree(node):
    if not node: return (BASE_INCLUDED, BASE_EXCLUDED)

    left_inc,  left_exc  = dp_tree(node.left)
    right_inc, right_exc = dp_tree(node.right)

    # If we include current node: children must be excluded
    include = node.val + left_exc + right_exc

    # If we exclude current node: children can be included OR excluded
    exclude = max(left_inc, left_exc) + max(right_inc, right_exc)

    return (include, exclude)


# Example: LC 337 — House Robber III
# Cannot rob two directly connected nodes
def rob_tree(root):
    def dfs(node):
        if not node: return (0, 0)          # (rob this node, skip this node)

        left_rob,  left_skip  = dfs(node.left)
        right_rob, right_skip = dfs(node.right)

        rob  = node.val + left_skip + right_skip   # rob here: children must be skipped
        skip = max(left_rob, left_skip) + max(right_rob, right_skip)  # skip here: children free

        return (rob, skip)

    return max(dfs(root))

**Practice problems:**

| Problem | State per node | Notes |
| :--- | :--- | :--- |
| LC 337 — House Robber III | `(rob, skip)` | Classic tree DP |
| LC 124 — Binary Tree Max Path Sum | max gain from this subtree | Global variable for cross-root answer |
| LC 543 — Diameter of Binary Tree | height of this subtree | Global variable updated at each node |

---
### Interval & Advanced DP — Summary

| Type | Signal words | Fill order | Problems |
| :--- | :--- | :--- | :--- |
| Interval DP | "burst", "merge", "score from splitting" | By increasing length | 312, 1312, 1039 |
| DP on Trees | "no adjacent nodes", "tree robber" | Postorder DFS | 337, 124, 543 |

---
# Part 5 — Decision Guide

```
Does the problem have overlapping subproblems and optimal substructure?
└── Yes → Use DP. Apply the 5-step framework first.

What is the shape of the input?
├── Single array or number
│   ├── Decisions at each index, small fixed window → Linear / Fibonacci DP  (70, 198)
│   ├── Max/min of a contiguous subarray            → Kadane's               (53, 152)
│   └── Longest subsequence with a condition        → LIS pattern            (300, 354)
│
├── Two strings
│   └── Comparing / aligning characters             → String DP 2D table     (1143, 72)
│
├── 2D grid
│   └── Paths from top-left to bottom-right         → Grid DP                (62, 64)
│
├── Set of items + a capacity / target
│   ├── Each item used at most once                 → 0/1 Knapsack (backwards) (416, 494)
│   └── Each item used unlimited times              → Unbounded Knapsack (forwards) (322, 518)
│
├── Contiguous range [i, j] (merge / split)
│   └── Split range at midpoint k                   → Interval DP (by length) (312, 1039)
│
└── Tree structure
    └── Choices at each node affecting parent        → DP on Trees (postorder DFS) (337)

Which implementation style?
├── State space is sparse (not all states reached)  → Top-down (memo)
└── Need space optimization or max performance      → Bottom-up (table)
```

---
## Full Pattern Cheat Sheet

| Pattern | Dimensions | Fill direction | Key trick | Problems |
| :--- | :--- | :--- | :--- | :--- |
| Linear / Fibonacci | 1D | Left → right | Can reduce to O(1) space | 70, 198, 746 |
| Kadane's | 1D | Left → right | Extend or start fresh | 53, 152, 918 |
| LIS | 1D | Left → right | Answer is `max(dp)`, not `dp[-1]` | 300, 354, 673 |
| Grid DP | 2D | Top-left → bottom-right | Init first row and col separately | 62, 63, 64 |
| String DP | 2D | Top-left → bottom-right | Extra row/col for empty string | 1143, 72, 583 |
| 0/1 Knapsack | 1D | **Backwards** | One item per use | 416, 494, 474 |
| Unbounded Knapsack | 1D | **Forwards** | Reuse same item | 322, 518, 139 |
| Interval DP | 2D | By increasing length | Split at midpoint k | 312, 1039 |
| DP on Trees | Per node | Postorder DFS | Return (include, exclude) tuple | 337, 124 |